In [28]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from  sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import r2_score

In [30]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_csv("../data/test_sample.csv")

X_train = train_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_train = train_data["LoyerMensuel_Log1"]

print(X_train)

# correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
# print(correlations.head(10))

     Chambres  Superficie_m2  DistanceRoute_m  AgeMaison  Salon_Bin  \
0         1.0      -1.592856         0.003290   0.040121          1   
1         4.0       0.699619         0.692082   0.305894          1   
2         4.0       0.216992        -1.084277   0.394485          1   
3         6.0       1.182245         1.223782  -1.554515          1   
4         3.0      -0.037727        -1.507220   1.280394          1   
..        ...            ...              ...        ...        ...   
377       4.0       0.243805         0.716250  -0.491424          1   
378       6.0       0.082929         1.163361   1.280394          1   
379       5.0       0.780056         1.682977  -0.314243          1   
380       6.0       1.638059        -1.567640   0.926030          1   
381       6.0       1.436964         0.982100   1.014621          1   

     SalleDeBainInterieure_Bin  Parking_Bin  Meuble_Bin  Jardin_Bin  \
0                            1            0           0           0   
1    

### Mise en niveau pour les données de test

In [32]:
# Les variables numériques à scaler
cols_a_scaler = ["Superficie_m2", "DistanceRoute_m", "AgeMaison"]

for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
    test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)

scaler = joblib.load("scaler.joblib")
test_data[cols_a_scaler] = scaler.transform(test_data[cols_a_scaler])

encoder = joblib.load("neighbourhood_encoder.joblib") 
test_data["Quartier_Target"] = encoder.transform(test_data[["Quartier"]])[:, 0]

test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Séparation de X_test et y_test
X_test = test_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_test = test_data["LoyerMensuel_Log1"]

X_test

,Chambres,Superficie_m2,DistanceRoute_m,AgeMaison,Salon_Bin,SalleDeBainInterieure_Bin,Parking_Bin,Meuble_Bin,Jardin_Bin,Quartier_Target
0,1.0,-1.352884,0.903554,-0.845788,1,1,1,0,0,0.002146
1,4.0,0.352396,-1.509637,0.491935,1,1,1,1,1,0.002182
2,3.0,-0.363500,0.977267,0.624821,1,1,1,0,1,0.002182
3,3.0,-0.095374,-0.988813,1.643617,1,0,1,0,0,0.002045
4,3.0,-0.214690,1.461838,-1.386193,1,0,0,0,0,0.002146
...,...,...,...,...,...,...,...,...,...,...
195,4.0,0.344352,0.106005,1.191803,0,1,1,0,1,0.002128
196,6.0,1.599180,-0.268602,-0.278806,1,1,0,0,0,0.010428
197,1.0,-1.622350,1.244325,-1.040688,1,1,1,0,0,0.002128
198,2.0,-0.934608,0.114463,-0.119343,1,1,0,0,0,0.001998


### Entrainement du modèle

In [56]:
boosting_model = GradientBoostingRegressor(
    learning_rate=0.1,
    n_estimators=160,
    max_depth=5,
    subsample=0.666,
    random_state=42
)

boosting_model.fit(X_train, y_train)

y_predict = boosting_model.predict(X_test)

y_predict_series = pd.Series(y_predict, index=y_test.index)

compareson = pd.DataFrame({
    "Valeurs réelles": np.expm1(y_test),
    "Valeurs prédites": np.expm1(y_predict_series)
})

compareson.head(5)

,Valeurs réelles,Valeurs prédites
0,923209.0,6.273160e+05
1,1091807.0,1.264089e+06
2,709703.0,7.875814e+05
3,782576.0,8.622550e+05
4,967262.0,8.956382e+05


### Mésure de performances avec cross_validate

In [57]:
scoring = ["neg_mean_squared_error", "neg_median_absolute_error", "neg_root_mean_squared_error", "r2"]

boosting_scores = cross_validate(boosting_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {boosting_scores["fit_time"]}")
print()
print(f"Score Time : {boosting_scores["score_time"]}")
print()
print(f"Test Neg Median Abosulte Error : {boosting_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg Mean Squared Error : {boosting_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test R2 : {boosting_scores["test_r2"]}")

print(r2_score(y_test, y_predict))

Fit Time : [0.28012252 0.22472262 0.21537161 0.20990157 0.20401907]

Score Time : [0.00784802 0.01047111 0.00549579 0.00514245 0.00566578]

Test Neg Median Abosulte Error : [-0.09723525 -0.14798811 -0.16609801 -0.17352915 -0.16635395]

Test Neg Mean Squared Error : [-0.1842798  -0.08995934 -0.06340749 -0.06885221 -0.07027688]

Test R2 : [0.59602384 0.8174486  0.8008411  0.76747945 0.79407066]
0.6723839989472342


### Sauvegarde du modèle

In [58]:

mean = {
    col: X_train[col].median() for col in X_train.select_dtypes(include=['number']).columns}
modes = {
    col: X_train[col].mode()[0] for col in X_train.select_dtypes(include=['object']).columns}

artefacts = {
    'boosting_model': boosting_model,
    'encoder': encoder,
    'mean': mean,
    'modes': modes,
    'columns': list(X_train.columns)
}

path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "data", "best_model_tuned.pkl")
joblib.dump(artefacts, path)

print(f"Modèle et prétraitements sauvegardés dans {path}")

Modèle et prétraitements sauvegardés dans /home/luckson/AI-Project/rent_prediction/data/best_model_tuned.pkl


In [60]:
importance = boosting_model.feature_importances_

print(X_train.columns)
print(importance)

classement = pd.DataFrame({
    "Variables": X_train.columns,
    "Feature_importance": importance
}).sort_values(ascending=False, by="Feature_importance")

classement

Index(['Chambres', 'Superficie_m2', 'DistanceRoute_m', 'AgeMaison',
       'Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin',
       'Jardin_Bin', 'Quartier_Target'],
      dtype='str')
[0.13869835 0.35933539 0.05893309 0.05999127 0.01295821 0.01185553
 0.00763323 0.03540151 0.0221128  0.29308061]


,Variables,Feature_importance
1,Superficie_m2,0.359335
9,Quartier_Target,0.293081
0,Chambres,0.138698
3,AgeMaison,0.059991
2,DistanceRoute_m,0.058933
7,Meuble_Bin,0.035402
8,Jardin_Bin,0.022113
4,Salon_Bin,0.012958
5,SalleDeBainInterieure_Bin,0.011856
6,Parking_Bin,0.007633
